In [321]:
import os
import shutil
import numpy as np
from google.colab import drive
drive.mount('/content/gdrive')
os.chdir('/content/gdrive/MyDrive/PQHIDE')

In [322]:
import hashlib

import numpy as np

from utils import gaussjordan

from LinearCode import LinearCode



class QC_MDPC(LinearCode):
    """
    Quasi-Cyclic LDPC code representation (extends LinearCode)

    ...

    Methods
    -------
    from_params(n, p, w)
        Init QC-LDPC by length, circulant size and code weight

    _get_circulant_block(polynom)
        Get circulant (p, p) for given vector of size p

    """

    def __init__(self, G, H):
        super().__init__(G, H)

    @classmethod
    def from_params(cls, n, p, w):
        assert n % p == 0, "p must be delimeter of n"

        n0 = n // p
        assert w > 2*n0, "not enough code weight"

        fine = False

        while not fine:
            blocks = []
            inverse_block = None
            inverse_block_position = None

            vector = [1 for _ in range(w)] + [0 for _  in range(n - w)]
            vector = np.array(vector, dtype=int)
            np.random.shuffle(vector)

            for i in range(n0):
                circ = vector[i*p:(i+1)*p]

                if sum(circ) < 2:
                    inverse_block = None
                    break

                block = QC_MDPC._get_circulant_block(circ)
                blocks.append(block)

                A, P = gaussjordan(block, True)
                A = np.array(A, dtype=int)
                P = np.array(P, dtype=int)

                if (A == np.eye(p, dtype=int)).all():
                    inverse_block_position = i
                    inverse_block = P

            # continue only if inverse circulant found
            fine = True if inverse_block is not None else False

        # put inverse block on last position
        blocks[inverse_block_position], blocks[n0-1] = blocks[n0-1], blocks[inverse_block_position]
        H = np.concatenate(blocks, axis=1)

        for i in range(n0):
            blocks[i] = blocks[i] @ inverse_block % 2
            blocks[i] = blocks[i].T

        Ht = np.concatenate(blocks[:n0-1], axis=0)
        G = np.concatenate((np.eye(Ht.shape[0], dtype=int), Ht), axis=1)

        assert (G @ H.T % 2 == 0).all(), "G is not correspond to H"

        return cls(G, H)

    @staticmethod
    def _get_circulant_block(polynom):
        N = len(polynom)
        block = np.empty((N, N), dtype=int)

        for i in range(N):
            block[i] = np.roll(polynom, i)

        return block



# EXAMPLE USAGE:

n = 8
p = 4
w = 5
errors_num = 2
qc_mdpc = QC_MDPC.from_params(n, p, w)

word = np.random.randint(2, size=qc_mdpc.getG().shape[0])
print(word)

encoded = qc_mdpc.encode(word)
print(encoded)
print(len(encoded))

# error vector size n with t or less errors
e = [1 for _ in range(errors_num)] + [0 for _  in range(n - errors_num)]
e = np.array(e, dtype=int)
np.random.shuffle(e)
print(e)
print(len(e))

corrupted = (encoded + e) % 2
print(corrupted)
print(len(corrupted))

decoded = qc_mdpc.decode(np.copy(corrupted))
decoded = qc_mdpc.get_message(decoded)
print(decoded)
print(len(decoded))

try:
    assert (decoded == word).all()
except AssertionError:
    print("The secret message must be reforwarded")

[1 1 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0
 1 0 1]
[1 1 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0
 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 0 1 1 0 1 0 0 0 0 1 0 0
 1 0 1 0 1 1]
80
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0]
80
[1 1 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 1
 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 0 1 1 0 1 0 0 0 0 1 1 0
 1 0 1 0 1 1]
80
[1 1 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0
 1 0 1]
40


In [323]:
os.chdir('/content/gdrive/MyDrive/PQHIDE/Species')

In [324]:
y = corrupted
print(y)
len(y)

[1 1 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 1
 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 0 1 1 0 1 0 0 0 0 1 1 0
 1 0 1 0 1 1]


80

In [325]:
folder_path = '/content/gdrive/MyDrive/PQHIDE/Species/file.txt'

folder_names = []

with open(folder_path, 'r') as f:
  for line in f:
    folder_names.append(line.strip()) # strip() removes potential newline characters


# Create the empty folders
i = 0
for folder_name in folder_names:
    if(i == len(y)):
        break
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    i = i + 1

In [326]:
count = 0
for folder_name in folder_names:
    if(os.path.isdir(os.path.join(folder_name))):
        count = count + 1

print(count)

80


In [327]:
ls

 Le volume dans le lecteur C n’a pas de nom.
 Le numéro de série du volume est 3C58-3F56

 Répertoire de C:\Users\Willy\Desktop\PQhide\PQHIDE\Species

04/03/2025  16:18    <DIR>          .
04/03/2025  16:22    <DIR>          ..
04/03/2025  16:18    <DIR>          Adriatic sturgeon
04/03/2025  16:18    <DIR>          African pompano
04/03/2025  16:18    <DIR>          Akiami paste shrimp
04/03/2025  16:18    <DIR>          Alamang shrimp
04/03/2025  16:18    <DIR>          Alexandria pompano
04/03/2025  16:18    <DIR>          American alligator
04/03/2025  16:18    <DIR>          Antarctic knobbed octopus
04/03/2025  16:18    <DIR>          Antarctic scallop
04/03/2025  16:18    <DIR>          Argentine seabass
04/03/2025  16:18    <DIR>          Atlantic deep-sea lobster
04/03/2025  16:18    <DIR>          Australian paste shrimp
04/03/2025  16:18    <DIR>          Aviu shrimp
04/03/2025  16:18    <DIR>          Babberlocks
04/03/2025  16:18    <DIR>          Baird's slickhead
04/03/2

In [328]:
value = 0
dico = {}
for filename in os.listdir('/content/gdrive/MyDrive/PQHIDE/Species'):
        if(os.path.isdir(filename)):
          dico[value]  = filename
          value = value + 1

In [329]:
 for i in dico.items():
    print(i)

(0, 'Adriatic sturgeon')
(1, 'African pompano')
(2, 'Akiami paste shrimp')
(3, 'Alamang shrimp')
(4, 'Alexandria pompano')
(5, 'American alligator')
(6, 'Antarctic knobbed octopus')
(7, 'Antarctic scallop')
(8, 'Argentine seabass')
(9, 'Atlantic deep-sea lobster')
(10, 'Australian paste shrimp')
(11, 'Aviu shrimp')
(12, 'Babberlocks')
(13, "Baird's slickhead")
(14, 'Beadlet anemone')
(15, 'Blackfin scad')
(16, 'Blackhead seabream')
(17, 'Bleak')
(18, 'Bonefish')
(19, 'Bonefishes nei')
(20, "Danube sturgeon'Green sturgeon")
(21, 'Deep-water mud lobster')
(22, 'Dovekie')
(23, 'European prickly cockle')
(24, 'Fingerprint oyster')
(25, 'Flat needlefish')
(26, 'Freshwater bream')
(27, 'Freshwater breams nei')
(28, 'Fringebarbel sturgeon')
(29, 'Girdle anemone')
(30, 'Glow-bellies splitfins nei')
(31, 'Glowbelly')
(32, 'Goldsilk seabream')
(33, 'Herring scad')
(34, 'Hooktooth dogfish')
(35, 'Indian threadfish')
(36, 'Jawla paste shrimp')
(37, 'Jembret shrimp')
(38, 'Koester')
(39, "Landlady'

In [330]:
def is_folder_empty(folder_path):
  """Checks if a folder is empty.

  Args:
    folder_path: The path to the folder.

  Returns:
    True if the folder is empty, False otherwise.
  """
  if not os.path.exists(folder_path):
    return True  # Non-existent folder is considered empty
  return not os.listdir(folder_path)


In [331]:
i = 0
while i < len(y):
  #number = np.random.randint(0,20)
  my_path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
  os.chdir(my_path)
  if os.path.exists(my_path):
    if is_folder_empty(my_path):
      with open(my_path + '/effective.txt','w') as fp:
          fp.write("0")
  i = i + 1

In [332]:
lis = []
i = 0

while i < len(y):
    print(dico[i])
    count = 0
    for filename in os.listdir('/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]):
        print(filename)
        count  = count + 1
    lis.append(count)
    print("-- nombre de fichiers  = ", lis[i],"--")
    print("\n")
    i = i + 1

Adriatic sturgeon
effective.txt
-- nombre de fichiers  =  1 --


African pompano
effective.txt
-- nombre de fichiers  =  1 --


Akiami paste shrimp
effective.txt
-- nombre de fichiers  =  1 --


Alamang shrimp
effective.txt
-- nombre de fichiers  =  1 --


Alexandria pompano
effective.txt
-- nombre de fichiers  =  1 --


American alligator
effective.txt
-- nombre de fichiers  =  1 --


Antarctic knobbed octopus
effective.txt
-- nombre de fichiers  =  1 --


Antarctic scallop
effective.txt
-- nombre de fichiers  =  1 --


Argentine seabass
effective.txt
-- nombre de fichiers  =  1 --


Atlantic deep-sea lobster
effective.txt
-- nombre de fichiers  =  1 --


Australian paste shrimp
effective.txt
-- nombre de fichiers  =  1 --


Aviu shrimp
effective.txt
-- nombre de fichiers  =  1 --


Babberlocks
effective.txt
-- nombre de fichiers  =  1 --


Baird's slickhead
effective.txt
-- nombre de fichiers  =  1 --


Beadlet anemone
effective.txt
-- nombre de fichiers  =  1 --


Blackfin scad
effe

In [333]:
lis

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [334]:
def hash_directory(path):
    digest = hashlib.sha256()

    for root, dirs, files in os.walk(path):
        for names in files:
            file_path = os.path.join(root, names)

            # Hash the path and add to the digest to account for empty files/directories
            digest.update(hashlib.sha1(file_path[len(path):].encode()).digest())

            # Per @pt12lol - if the goal is uniqueness over repeatability, this is an alternative method using 'hash'
            # digest.update(str(hash(file_path[len(path):])).encode())

            if os.path.isfile(file_path):
                with open(file_path, 'rb') as f_obj:
                    while True:
                        buf = f_obj.read(1024 * 1024)
                        if not buf:
                            break
                        digest.update(buf)

    return digest.hexdigest()

In [335]:
list_Hash1 = []
for i in range(len(y)):
    path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
    list_Hash1.insert(i,hash_directory(path))
    print('--hash '+dico[i],'--',list_Hash1[i])

--hash Adriatic sturgeon -- fd14416da1560f8ef421d2aadd287e7d39ad0ba69548265fca3020b971bdfc10
--hash African pompano -- 12ff5a4e77e00fbdb027f823c81e01dfa63e4a814f890aae05e3422dbebf825d
--hash Akiami paste shrimp -- feb9555d1c70ae4cfb5a6e15e2674b5ca5998d83ca628c3e8c81b3a6e4622b5b
--hash Alamang shrimp -- 4a4bbfca14086394c7401d005e5b1e9c80c8004a5ea858971b943c229e47bbed
--hash Alexandria pompano -- 4a4bbfca14086394c7401d005e5b1e9c80c8004a5ea858971b943c229e47bbed
--hash American alligator -- 599d89c28cef4ed87f94127d87452ec8462e5e9aecbb57cffabdeb88996ebe32
--hash Antarctic knobbed octopus -- 599d89c28cef4ed87f94127d87452ec8462e5e9aecbb57cffabdeb88996ebe32
--hash Antarctic scallop -- 599d89c28cef4ed87f94127d87452ec8462e5e9aecbb57cffabdeb88996ebe32
--hash Argentine seabass -- 12ff5a4e77e00fbdb027f823c81e01dfa63e4a814f890aae05e3422dbebf825d
--hash Atlantic deep-sea lobster -- e9a255f20e1ddcb9045fa4e800a34f5858b2bd2c85cb2333a799b1924118d2a5
--hash Australian paste shrimp -- feb9555d1c70ae4cfb5a6

In [336]:
i = 0
while(i < len(y)):
    if(y[i] == 1):
      my_path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
      os.chdir(my_path)
      
      if os.path.exists(my_path):
        try:
            for filename in os.listdir(my_path):
                file_path = os.path.join(my_path, filename)
                with open(file_path,'r') as fp:
                    content = fp.read()
                    new_content = int(content) + 1
                fp.close()
                with open(file_path,'w') as fp:
                    fp.write(str(new_content))
                fp.close()
        except Exception as e:
              print('Failed to open %s. Reason: %s' % (my_path, e))
    i = i + 1

In [337]:
lis = []
i = 0

while i < len(y):
    print(dico[i])
    count = 0
    for filename in os.listdir('/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]):
        print(filename)
        count  = count + 1
    lis.append(count)
    print("-- nombre de fichiers  = ", lis[i],"--")
    print("\n")
    i = i + 1

Adriatic sturgeon
effective.txt
-- nombre de fichiers  =  1 --


African pompano
effective.txt
-- nombre de fichiers  =  1 --


Akiami paste shrimp
effective.txt
-- nombre de fichiers  =  1 --


Alamang shrimp
effective.txt
-- nombre de fichiers  =  1 --


Alexandria pompano
effective.txt
-- nombre de fichiers  =  1 --


American alligator
effective.txt
-- nombre de fichiers  =  1 --


Antarctic knobbed octopus
effective.txt
-- nombre de fichiers  =  1 --


Antarctic scallop
effective.txt
-- nombre de fichiers  =  1 --


Argentine seabass
effective.txt
-- nombre de fichiers  =  1 --


Atlantic deep-sea lobster
effective.txt
-- nombre de fichiers  =  1 --


Australian paste shrimp
effective.txt
-- nombre de fichiers  =  1 --


Aviu shrimp
effective.txt
-- nombre de fichiers  =  1 --


Babberlocks
effective.txt
-- nombre de fichiers  =  1 --


Baird's slickhead
effective.txt
-- nombre de fichiers  =  1 --


Beadlet anemone
effective.txt
-- nombre de fichiers  =  1 --


Blackfin scad
effe

In [338]:
list_Hash2 = []
for i in range(len(y)):
    path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
    list_Hash2.insert(i,hash_directory(path))
    print('--hash '+dico[i],'--',list_Hash2[i])

--hash Adriatic sturgeon -- 928ef66f25ef9557defee1a80939c77c24fcb458c7280f4de7ca74e99ff8cf20
--hash African pompano -- e9a255f20e1ddcb9045fa4e800a34f5858b2bd2c85cb2333a799b1924118d2a5
--hash Akiami paste shrimp -- feb9555d1c70ae4cfb5a6e15e2674b5ca5998d83ca628c3e8c81b3a6e4622b5b
--hash Alamang shrimp -- feb9555d1c70ae4cfb5a6e15e2674b5ca5998d83ca628c3e8c81b3a6e4622b5b
--hash Alexandria pompano -- 4a4bbfca14086394c7401d005e5b1e9c80c8004a5ea858971b943c229e47bbed
--hash American alligator -- 599d89c28cef4ed87f94127d87452ec8462e5e9aecbb57cffabdeb88996ebe32
--hash Antarctic knobbed octopus -- 599d89c28cef4ed87f94127d87452ec8462e5e9aecbb57cffabdeb88996ebe32
--hash Antarctic scallop -- 12ff5a4e77e00fbdb027f823c81e01dfa63e4a814f890aae05e3422dbebf825d
--hash Argentine seabass -- 12ff5a4e77e00fbdb027f823c81e01dfa63e4a814f890aae05e3422dbebf825d
--hash Atlantic deep-sea lobster -- e9a255f20e1ddcb9045fa4e800a34f5858b2bd2c85cb2333a799b1924118d2a5
--hash Australian paste shrimp -- feb9555d1c70ae4cfb5a6

In [339]:
msg_secret = []
for i in range(len(y)):
    if(list_Hash1[i] != list_Hash2[i]):
        msg_secret.append(1)
    else:
        msg_secret.append(0)

In [340]:
np.array(msg_secret)

array([1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1])

In [341]:
y

array([1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1], dtype=int32)

In [342]:
len(msg_secret)

80

In [343]:
decoded = qc_mdpc.decode(np.copy(msg_secret))
decoded = qc_mdpc.get_message(decoded)
print(decoded)
print(len(decoded))

[1 1 0 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0
 1 0 1]
40


In [344]:
word

array([1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1])

In [345]:
try:
    assert (msg_secret == y).all()
    assert (decoded == word).all()
except AssertionError:
    print("The secret message must be reforwarded")
else:
    print("The secret message has been correctly forwarded")

The secret message has been correctly forwarded


# #Code script for time evaluation

In [362]:
import hashlib

import numpy as np

from utils import gaussjordan

from LinearCode import LinearCode

import time



class QC_MDPC(LinearCode):
    """
    Quasi-Cyclic LDPC code representation (extends LinearCode)

    ...

    Methods
    -------
    from_params(n, p, w)
        Init QC-LDPC by length, circulant size and code weight

    _get_circulant_block(polynom)
        Get circulant (p, p) for given vector of size p

    """

    def __init__(self, G, H):
        super().__init__(G, H)

    @classmethod
    def from_params(cls, n, p, w):
        assert n % p == 0, "p must be delimeter of n"

        n0 = n // p
        assert w > 2*n0, "not enough code weight"

        fine = False

        while not fine:
            blocks = []
            inverse_block = None
            inverse_block_position = None

            vector = [1 for _ in range(w)] + [0 for _  in range(n - w)]
            vector = np.array(vector, dtype=int)
            np.random.shuffle(vector)

            for i in range(n0):
                circ = vector[i*p:(i+1)*p]

                if sum(circ) < 2:
                    inverse_block = None
                    break

                block = QC_MDPC._get_circulant_block(circ)
                blocks.append(block)

                A, P = gaussjordan(block, True)
                A = np.array(A, dtype=int)
                P = np.array(P, dtype=int)

                if (A == np.eye(p, dtype=int)).all():
                    inverse_block_position = i
                    inverse_block = P

            # continue only if inverse circulant found
            fine = True if inverse_block is not None else False

        # put inverse block on last position
        blocks[inverse_block_position], blocks[n0-1] = blocks[n0-1], blocks[inverse_block_position]
        H = np.concatenate(blocks, axis=1)

        for i in range(n0):
            blocks[i] = blocks[i] @ inverse_block % 2
            blocks[i] = blocks[i].T

        Ht = np.concatenate(blocks[:n0-1], axis=0)
        G = np.concatenate((np.eye(Ht.shape[0], dtype=int), Ht), axis=1)

        assert (G @ H.T % 2 == 0).all(), "G is not correspond to H"

        return cls(G, H)

    @staticmethod
    def _get_circulant_block(polynom):
        N = len(polynom)
        block = np.empty((N, N), dtype=int)

        for i in range(N):
            block[i] = np.roll(polynom, i)

        return block



# EXAMPLE USAGE:

n = 8
p = 4
w = 5
errors_num = 2
qc_mdpc = QC_MDPC.from_params(n, p, w)

start = time.time()

word = np.random.randint(2, size=qc_mdpc.getG().shape[0])
print(word)

encoded = qc_mdpc.encode(word)
print(encoded)
print(len(encoded))

# error vector size n with t or less errors
e = [1 for _ in range(errors_num)] + [0 for _  in range(n - errors_num)]
e = np.array(e, dtype=int)
np.random.shuffle(e)
print(e)
print(len(e))

corrupted = (encoded + e) % 2
print(corrupted)
print(len(corrupted))

os.chdir('/content/gdrive/MyDrive/PQHIDE/Species')

y = corrupted

#liste = ['Balistidae', 'Belonidae', 'Enoploteuthidae', 'Cyprinidae', 'Nephropidae', 'Axiidae', 'Serranidae', 'Cardiidae', 'Scombridae', 'Sparidae', 'Acanthuridae', 'Limidae', 'Sergestidae', 'Acipenseridae', 'Acropomatidae', 'Holothuriidae', 'Squalidae', 'Octopodidae', 'Pectinidae', 'Serranidae']

i = 0
while i < len(y):
  #number = np.random.randint(0,20)
  my_path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
  os.chdir(my_path)
  if os.path.exists(my_path):
    if is_folder_empty(my_path):
      with open(my_path + '/effective.txt','w') as fp:
          fp.write("0")
      fp.close
  i = i + 1

list_Hash1 = []
for i in range(len(y)):
    path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
    list_Hash1.insert(i,hash_directory(path))

i = 0
while(i < len(y)):
    if(y[i] == 1):
      my_path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
      os.chdir(my_path)
      
      if os.path.exists(my_path):
        try:
            for filename in os.listdir(my_path):
                file_path = os.path.join(my_path, filename)
                with open(file_path,'r') as fp:
                    content = fp.read()
                    new_content = int(content) + 1
                fp.close()
                with open(file_path,'w') as fp:
                    fp.write(str(new_content))
                fp.close()
        except Exception as e:
              print('Failed to open %s. Reason: %s' % (my_path, e))
    i = i + 1

end = time.time()
print('time Encryption = ',end - start)
    
    
start = time.time()

list_Hash2 = []
for i in range(len(y)):
    path = '/content/gdrive/MyDrive/PQHIDE/Species/' + dico[i]
    list_Hash2.insert(i,hash_directory(path))

msg_secret = []
for i in range(len(y)):
    if(list_Hash1[i] != list_Hash2[i]):
        msg_secret.append(1)
    else:
        msg_secret.append(0)

print("Secret =",msg_secret)
#print(len(msg_secret))
decoded = qc_mdpc.decode(np.copy(msg_secret))
decoded = qc_mdpc.get_message(decoded)
print(decoded)

end = time.time()
print('time Decryption = ',end - start)


try:
    assert (msg_secret == y).all()
    assert (decoded == word).all()
except AssertionError:
    print("The secret message must be reforwarded")
else:
    print("The secret message has been correctly forwarded")

[0 1 1 0 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0
 0 0 1]
[0 1 1 0 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0
 0 0 1 1 0 0 0 1 1 1 1 0 1 1 1 0 0 0 0 0 1 0 0 1 1 0 0 0 0 1 1 0 1 0 1 1 0
 1 0 0 0 1 0]
80
[0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0]
80
[0 1 1 1 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0
 0 0 1 1 0 0 0 1 1 1 1 0 1 1 1 0 0 0 0 0 1 0 0 1 1 0 0 0 0 1 1 0 1 0 1 1 0
 0 0 0 0 1 0]
80
time Encryption =  0.044509172439575195
Secret = [0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0]
[0 1 1 0 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0
 0 0 1]
time Decryption =  0.03400015830993652
The 